## Setup

In [7]:
from pathlib import Path
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks, regularizers
import kagglehub

# Configuration
COMPETITION_NAME = 'competition-2026-fait-main'
CLASS_NAMES = ['coastguard_scaled', 'containership_scaled', 'corvette_scaled', 'cruiser_scaled',
               'cv_scaled', 'destroyer_scaled', 'methanier_scaled', 'smallfish_scaled', 'submarine_scaled', 'tug_scaled']
IMAGE_SIZE = (128, 192)
BATCH_SIZE = 32
EPOCHS = 70
MODEL_PATH = Path('ship_classifier.h5')
SUBMISSION_PATH = Path('test.csv')

In [8]:
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)

cache_root = Path.home() / '.cache' / 'kagglehub' / 'competitions' / COMPETITION_NAME
if cache_root.exists():
    competition_dir = cache_root
else:
    print('test')
    competition_dir = Path(kagglehub.competition_download(COMPETITION_NAME))
    print(f'Dataset {competition_dir}')

train_dir = competition_dir / 'ships24' / 'ships_gray' / 'ships_gray'

In [9]:
common_kwargs = dict(labels='inferred', label_mode='int', class_names=CLASS_NAMES,
                     image_size=IMAGE_SIZE, color_mode='grayscale', batch_size=BATCH_SIZE,
                     validation_split=0.15, seed=42)

train_ds = tf.keras.utils.image_dataset_from_directory(train_dir, subset='training', **common_kwargs)
val_ds = tf.keras.utils.image_dataset_from_directory(train_dir, subset='validation', **common_kwargs)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(1000).prefetch(AUTOTUNE)
val_ds = val_ds.cache().prefetch(AUTOTUNE)

counts = [len(list((train_dir / cls).iterdir())) for cls in CLASS_NAMES]
total = sum(counts)
class_weight = {i: total / (len(CLASS_NAMES) * c) if c > 0 else 1.0 for i, c in enumerate(counts)}

print(f'Train batches {len(train_ds)}, Val batches {len(val_ds)}')
print(f'Class weights {class_weight}')

Found 37980 files belonging to 10 classes.
Using 32283 files for training.
Found 37980 files belonging to 10 classes.
Using 5697 files for validation.
Train batches 1009, Val batches 179
Class weights {0: 1.2141943734015346, 1: 0.6061283115225023, 2: 1.4192825112107623, 3: 0.6075827867541194, 4: 2.0799561883899234, 5: 0.6350108677478683, 6: 1.1973518284993694, 7: 1.2601194426011944, 8: 1.4824355971896956, 9: 1.2235824742268042}


## Modele

In [10]:
model = models.Sequential([
    layers.Input(shape=(128, 192, 1)),
    layers.Rescaling(1.0 / 255.0),
    
    layers.RandomFlip('horizontal'),
    layers.RandomTranslation(height_factor=0.05, width_factor=0.05),
    layers.RandomZoom(0.10),
    layers.RandomContrast(0.15),
    
    layers.Conv2D(64, 3, padding='same', activation='swish'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(2),
    
    layers.Conv2D(128, 3, padding='same', activation='swish'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(2),
    
    layers.Conv2D(256, 3, padding='same', activation='swish'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(2),
    
    layers.Conv2D(512, 3, padding='same', activation='swish'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(2),
    
    layers.GlobalAveragePooling2D(),
    layers.Dense(256, activation='swish'),
    layers.Dropout(0.4),
    layers.Dense(len(CLASS_NAMES), activation='softmax'),
])

model.summary()
print(f'Nombre de couches: {len(model.layers)}')

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ rescaling_1 (Rescaling)         │ (None, 128, 192, 1)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_flip_1 (RandomFlip)      │ (None, 128, 192, 1)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_translation_1            │ (None, 128, 192, 1)    │             0 │
│ (RandomTranslation)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_zoom_1 (RandomZoom)      │ (None, 128, 192, 1)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_contrast_1               │ (None, 128, 192, 1)    │             0 │
│ (RandomContrast)                │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 128, 192, 64)   │           640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 128, 192, 64)   │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 64, 96, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 64, 96, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 64, 96, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 32, 48, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_6 (Conv2D)               │ (None, 32, 48, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_6           │ (None, 32, 48, 256)    │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_6 (MaxPooling2D)  │ (None, 16, 24, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_7 (Conv2D)               │ (None, 16, 24, 512)    │     1,180,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_7           │ (None, 16, 24, 512)    │         2,048 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_7 (MaxPooling2D)  │ (None, 8, 12, 512)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 512)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 10)             │         2,570 │
└─────────────────────────────────┴────────────────────────┴─────────────

 Total params: 1,687,562 (6.44 MB)

 Trainable params: 1,685,642 (6.43 MB)

 Non-trainable params: 1,920 (7.50 KB)

Nombre de couches: 21


In [11]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3), # Modifié ici
    loss='sparse_categorical_crossentropy',
    metrics=['sparse_categorical_accuracy'],
)

callbacks_list = [
    callbacks.ModelCheckpoint(str(MODEL_PATH), save_best_only=True, 
                             monitor='val_sparse_categorical_accuracy', mode='max', verbose=1),
    callbacks.EarlyStopping(monitor='val_sparse_categorical_accuracy', patience=10, 
                           restore_best_weights=True, mode='max', verbose=1),
    callbacks.ReduceLROnPlateau(monitor='val_sparse_categorical_accuracy', factor=0.5, 
                               patience=4, min_lr=1e-6, verbose=1),
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=60,
    class_weight=class_weight,
    callbacks=callbacks_list,
)

Epoch 1/60


I0000 00:00:1781286976.005699    3073 shuffle_dataset_op.cc:453] ShuffleDatasetV3:34: Filling up shuffle buffer (this may take a while): 864 of 1000
I0000 00:00:1781286976.649984    3073 shuffle_dataset_op.cc:483] Shuffle buffer filled.
I0000 00:00:1781286979.716067    2970 cuda_dnn.cc:461] Loaded cuDNN version 92200


1009/1009 ━━━━━━━━━━━━━━━━━━━━ 0s 136ms/step - loss: 1.9923 - sparse_categorical_accuracy: 0.3331
Epoch 1: val_sparse_categorical_accuracy improved from None to 0.25241, saving model to ship_classifier.h5



Epoch 1: finished saving model to ship_classifier.h5
1009/1009 ━━━━━━━━━━━━━━━━━━━━ 166s 142ms/step - loss: 1.6744 - sparse_categorical_accuracy: 0.4525 - val_loss: 3.6798 - val_sparse_categorical_accuracy: 0.2524 - learning_rate: 0.0010
Epoch 2/60
1009/1009 ━━━━━━━━━━━━━━━━━━━━ 0s 134ms/step - loss: 1.0563 - sparse_categorical_accuracy: 0.6656
Epoch 2: val_sparse_categorical_accuracy improved from 0.25241 to 0.65701, saving model to ship_classifier.h5



Epoch 2: finished saving model to ship_classifier.h5
1009/1009 ━━━━━━━━━━━━━━━━━━━━ 141s 140ms/step - loss: 0.9675 - sparse_categorical_accuracy: 0.6947 - val_loss: 0.9486 - val_sparse_categorical_accuracy: 0.6570 - learning_rate: 0.0010
Epoch 3/60
1009/1009 ━━━━━━━━━━━━━━━━━━━━ 0s 136ms/step - loss: 0.7398 - sparse_categorical_accuracy: 0.7671
Epoch 3: val_sparse_categorical_accuracy did not improve from 0.65701
1009/1009 ━━━━━━━━━━━━━━━━━━━━ 143s 142ms/step - loss: 0.7063 - sparse_categorical_accuracy: 0.7777 - val_loss: 1.2661 - val_sparse_categorical_accuracy: 0.6247 - learning_rate: 0.0010
Epoch 4/60
1009/1009 ━━━━━━━━━━━━━━━━━━━━ 0s 134ms/step - loss: 0.5914 - sparse_categorical_accuracy: 0.8131
Epoch 4: val_sparse_categorical_accuracy improved from 0.65701 to 0.66403, saving model to ship_classifier.h5


...


Epoch 39: finished saving model to ship_classifier.h5
1009/1009 ━━━━━━━━━━━━━━━━━━━━ 244s 242ms/step - loss: 0.0421 - sparse_categorical_accuracy: 0.9858 - val_loss: 0.3687 - val_sparse_categorical_accuracy: 0.9212 - learning_rate: 2.5000e-04
Epoch 40/60
1009/1009 ━━━━━━━━━━━━━━━━━━━━ 0s 283ms/step - loss: 0.0368 - sparse_categorical_accuracy: 0.9869
Epoch 40: val_sparse_categorical_accuracy did not improve from 0.92119
1009/1009 ━━━━━━━━━━━━━━━━━━━━ 297s 295ms/step - loss: 0.0341 - sparse_categorical_accuracy: 0.9878 - val_loss: 0.3992 - val_sparse_categorical_accuracy: 0.9152 - learning_rate: 2.5000e-04
Epoch 41/60
1009/1009 ━━━━━━━━━━━━━━━━━━━━ 0s 201ms/step - loss: 0.0357 - sparse_categorical_accuracy: 0.9877
Epoch 41: val_sparse_categorical_accuracy did not improve from 0.92119
1009/1009 ━━━━━━━━━━━━━━━━━━━━ 209s 207ms/step - loss: 0.0345 - sparse_categorical_accuracy: 0.9881 - val_loss: 0.4324 - val_sparse_categorical_accuracy: 0.9078 - learning_rate: 2.5000e-04
Epoch 42/60
1009


Epoch 45: finished saving model to ship_classifier.h5
1009/1009 ━━━━━━━━━━━━━━━━━━━━ 142s 141ms/step - loss: 0.0213 - sparse_categorical_accuracy: 0.9931 - val_loss: 0.3722 - val_sparse_categorical_accuracy: 0.9245 - learning_rate: 1.2500e-04
Epoch 46/60
1009/1009 ━━━━━━━━━━━━━━━━━━━━ 0s 159ms/step - loss: 0.0235 - sparse_categorical_accuracy: 0.9922
Epoch 46: val_sparse_categorical_accuracy did not improve from 0.92452
1009/1009 ━━━━━━━━━━━━━━━━━━━━ 166s 165ms/step - loss: 0.0230 - sparse_categorical_accuracy: 0.9922 - val_loss: 0.4067 - val_sparse_categorical_accuracy: 0.9182 - learning_rate: 1.2500e-04
Epoch 47/60
1009/1009 ━━━━━━━━━━━━━━━━━━━━ 0s 137ms/step - loss: 0.0210 - sparse_categorical_accuracy: 0.9921
Epoch 47: val_sparse_categorical_accuracy did not improve from 0.92452
1009/1009 ━━━━━━━━━━━━━━━━━━━━ 144s 143ms/step - loss: 0.0211 - sparse_categorical_accuracy: 0.9924 - val_loss: 0.3866 - val_sparse_categorical_accuracy: 0.9189 - learning_rate: 1.2500e-04
Epoch 48/60
1009


Epoch 50: finished saving model to ship_classifier.h5
1009/1009 ━━━━━━━━━━━━━━━━━━━━ 143s 141ms/step - loss: 0.0175 - sparse_categorical_accuracy: 0.9941 - val_loss: 0.3799 - val_sparse_categorical_accuracy: 0.9277 - learning_rate: 6.2500e-05
Epoch 51/60
1009/1009 ━━━━━━━━━━━━━━━━━━━━ 0s 135ms/step - loss: 0.0190 - sparse_categorical_accuracy: 0.9928
Epoch 51: val_sparse_categorical_accuracy did not improve from 0.92768
1009/1009 ━━━━━━━━━━━━━━━━━━━━ 142s 141ms/step - loss: 0.0179 - sparse_categorical_accuracy: 0.9937 - val_loss: 0.3862 - val_sparse_categorical_accuracy: 0.9270 - learning_rate: 6.2500e-05
Epoch 52/60
1009/1009 ━━━━━━━━━━━━━━━━━━━━ 0s 135ms/step - loss: 0.0160 - sparse_categorical_accuracy: 0.9946
Epoch 52: val_sparse_categorical_accuracy did not improve from 0.92768
1009/1009 ━━━━━━━━━━━━━━━━━━━━ 146s 145ms/step - loss: 0.0167 - sparse_categorical_accuracy: 0.9943 - val_loss: 0.3844 - val_sparse_categorical_accuracy: 0.9270 - learning_rate: 6.2500e-05
Epoch 53/60
1009


Epoch 53: finished saving model to ship_classifier.h5
1009/1009 ━━━━━━━━━━━━━━━━━━━━ 143s 142ms/step - loss: 0.0166 - sparse_categorical_accuracy: 0.9947 - val_loss: 0.3793 - val_sparse_categorical_accuracy: 0.9287 - learning_rate: 6.2500e-05
Epoch 54/60
1009/1009 ━━━━━━━━━━━━━━━━━━━━ 0s 135ms/step - loss: 0.0177 - sparse_categorical_accuracy: 0.9939
Epoch 54: val_sparse_categorical_accuracy did not improve from 0.92873
1009/1009 ━━━━━━━━━━━━━━━━━━━━ 142s 140ms/step - loss: 0.0162 - sparse_categorical_accuracy: 0.9944 - val_loss: 0.3771 - val_sparse_categorical_accuracy: 0.9282 - learning_rate: 6.2500e-05
Epoch 55/60
1009/1009 ━━━━━━━━━━━━━━━━━━━━ 0s 134ms/step - loss: 0.0146 - sparse_categorical_accuracy: 0.9947
Epoch 55: val_sparse_categorical_accuracy did not improve from 0.92873
1009/1009 ━━━━━━━━━━━━━━━━━━━━ 141s 140ms/step - loss: 0.0154 - sparse_categorical_accuracy: 0.9942 - val_loss: 0.3811 - val_sparse_categorical_accuracy: 0.9286 - learning_rate: 6.2500e-05
Epoch 56/60
1009


Epoch 58: finished saving model to ship_classifier.h5
1009/1009 ━━━━━━━━━━━━━━━━━━━━ 139s 138ms/step - loss: 0.0150 - sparse_categorical_accuracy: 0.9953 - val_loss: 0.3895 - val_sparse_categorical_accuracy: 0.9298 - learning_rate: 3.1250e-05
Epoch 59/60
1009/1009 ━━━━━━━━━━━━━━━━━━━━ 0s 136ms/step - loss: 0.0145 - sparse_categorical_accuracy: 0.9952
Epoch 59: val_sparse_categorical_accuracy did not improve from 0.92979
1009/1009 ━━━━━━━━━━━━━━━━━━━━ 143s 142ms/step - loss: 0.0143 - sparse_categorical_accuracy: 0.9952 - val_loss: 0.3920 - val_sparse_categorical_accuracy: 0.9286 - learning_rate: 3.1250e-05
Epoch 60/60
1009/1009 ━━━━━━━━━━━━━━━━━━━━ 0s 153ms/step - loss: 0.0137 - sparse_categorical_accuracy: 0.9952
Epoch 60: val_sparse_categorical_accuracy did not improve from 0.92979
1009/1009 ━━━━━━━━━━━━━━━━━━━━ 165s 163ms/step - loss: 0.0132 - sparse_categorical_accuracy: 0.9953 - val_loss: 0.3946 - val_sparse_categorical_accuracy: 0.9259 - learning_rate: 3.1250e-05
Restoring model 

## Evaluer

In [12]:
val_loss, val_acc = model.evaluate(val_ds)
print(f'Validation Loss {val_loss:.4f}, Accuracy {val_acc:.4f}')

model.save(MODEL_PATH)
print(f'{MODEL_PATH}')

test_path = competition_dir / 'ships24' / 'test_images.npy'
test_images = np.load(test_path).astype(np.float32)
test_images = np.expand_dims(test_images, axis=-1)
print(f'Test images shape {test_images.shape}')

predictions = model.predict(test_images, batch_size=BATCH_SIZE)
predicted_labels = np.argmax(predictions, axis=1)

unique, counts = np.unique(predicted_labels, return_counts=True)
print(f'distribution {dict(zip(unique.tolist(), counts.tolist()))}')

179/179 ━━━━━━━━━━━━━━━━━━━━ 7s 39ms/step - loss: 0.3895 - sparse_categorical_accuracy: 0.9298


Validation Loss 0.3895, Accuracy 0.9298
ship_classifier.h5
Test images shape (4224, 128, 192, 1)


W0000 00:00:1781297141.025064    2776 cpu_allocator_impl.cc:82] Allocation of 415236096 exceeds 10% of free system memory.
W0000 00:00:1781297143.386962    2776 cpu_allocator_impl.cc:82] Allocation of 415236096 exceeds 10% of free system memory.


132/132 ━━━━━━━━━━━━━━━━━━━━ 6s 40ms/step
distribution {0: 362, 1: 689, 2: 302, 3: 701, 4: 199, 5: 661, 6: 354, 7: 328, 8: 294, 9: 334}


In [13]:
SUBMISSION_PATH.parent.mkdir(parents=True, exist_ok=True)
with open(SUBMISSION_PATH, 'w') as f:
    f.write('ID,Category\n')
    for idx, label in enumerate(predicted_labels):
        f.write(f'{idx},{int(label)}\n')
print(f'{SUBMISSION_PATH}')

test.csv
